# Chemprop-IR Data generation

---

### What this notebook does

`chemprop-IR` is the *spectrum-simulation* front-end of MMST: it learns the
inverse mapping "IR spectrum -> molecular descriptors" and is used to attach an
IR modality to molecules that only have a SMILES string.

This notebook uses it in the **forward-free** direction: for every SMILES in a
CSV, `chemprop.predict` emits a 1000-point IR absorbance vector, one `.csv` file
per molecule, plus a small table with the molecular formula. These per-molecule
IR files are exactly what `config.IR_data_folder` points at in
`1.0_Experiment_Notebook___.ipynb` (`data/IBM_dataset/IBM_ir_sim`).

| Cell | Role |
| --- | --- |
| 1-2 | put `chemprop-IR/` on `sys.path`, import chemprop, list feature generators |
| 3 | chemprop argument bundle -> `Namespace` via `parse_arguments` |
| 4 | `modify_predict_args` (fills derived fields such as `spectra_type_nr`) |
| 5 | section header |
| 6 | *(duplicate of cells 1-2)* same imports, re-run for a standalone kernel |
| 7 | duplicate config + two helper functions (`generate_spectral_data_batch`, `calculate_molecular_formulas`) |
| 8 | main loop: read SMILES in batches of 64, predict IR, write one CSV per molecule |
| 9 | write the summary table `chemprop-ir-sim-IBM.csv` |
| 10 | sanity check: how many per-molecule files were produced |

**Reading caveat:** cells 1-4 and 6-7 duplicate each other, and cell 5 says
"big ZINC dataset" while the code actually processes `IBM_SMI_data.csv`. Both
artefacts are noted in place; the notebook is kept byte-for-byte executable so
the stored outputs stay valid.

In [ ]:
# =============================================================================
# CELL 1 -- Environment bootstrap
#
# chemprop-IR is vendored as a *sub-directory* of this repository, not installed
# as a package, so its importable root must be appended to sys.path. Without
# this block `from chemprop.train.predict import predict` raises ModuleNotFoundError.
#
# NOTE (known issue, left unchanged): sys.path.append(...) is executed on every
# run of the cell, so sys.path accumulates duplicate entries in a long-lived
# kernel. It is harmless (Python simply tries the same directory twice), but
# `sys.path.insert(0, ...)` guarded by `if str(path) not in sys.path` would be
# the clean form.
# =============================================================================

import os
import sys
from pathlib import Path

# Get the current working directory
# `Path.cwd()` == the directory Jupyter was started in; all paths below are relative to it.
notebook_dir = Path.cwd()

# Add the chemprop-IR directory to the path
chemprop_ir_path = notebook_dir / "chemprop-IR"
sys.path.append(str(chemprop_ir_path))

from argparse import Namespace   # attribute container for the chemprop args
import csv                       # only used by legacy paths -- the current cells use pandas
from typing import List, Optional  # type hints for the helper functions
import numpy as np               # arrays holding the predicted spectra
import torch                     # chemprop is a PyTorch model; used for the device list
from tqdm import tqdm            # progress bar over the molecule batches

In [ ]:
# =============================================================================
# CELL 2 -- Import the chemprop pieces used later
#
# Layer by layer:
#   chemprop.train.predict.predict        -> the low-level "run a checkpoint on a
#                                            dataset" function (used inside
#                                            make_predictions)
#   chemprop.train.make_predictions       -> the high-level entry point called in
#                                            cell 7; takes (args, smiles=...)
#   chemprop.parsing.modify_predict_args  -> fills derived args (spectra_type_nr,
#                                            feature sizes, device flags, ...)
#   chemprop.data.utils.get_data          -> CSV reader used when args.test_path is
#                                            set (not used here, we pass SMILES directly)
#   chemprop.utils.load_args/load_checkpoint/load_scalers
#                                         -> manual checkpoint loading, imported
#                                            for interactive debugging of the
#                                            ensemble in checkpoint_dir
#   roundrobin_sid                        -> spectral information divergence loss,
#                                            the IR-specific objective (not called
#                                            in this notebook, imported for reference)
#
# `get_available_features_generators()` is called only to *print* the registered
# feature generators -- it has no effect on the prediction (config sets
# features_generator=None). Its output is the list of extra-descriptor modes you
# could switch on.
# =============================================================================

from chemprop.train.predict import predict
from chemprop.data import MoleculeDataset
from chemprop.data.utils import get_data, get_data_from_smiles
from chemprop.utils import load_args, load_checkpoint, load_scalers
from chemprop.train.spectral_loss import roundrobin_sid
from chemprop.features import get_available_features_generators
get_available_features_generators()   # display-only call: prints the feature generator registry

In [ ]:
# =============================================================================
# CELL 3 -- chemprop argument bundle
#
# The keys mirror chemprop's `predict` CLI. Every value is a ONE-element list
# because chemprop supports checkpoint *ensembles*: one entry per model in the
# ensemble. `parse_arguments` below unwraps them into a Namespace.
#
# Field glossary:
#   gpu                -- visible device ids; all GPUs are claimed by default
#   test_path          -- input CSV of spectra; unused here because cell 7 passes
#                         `smiles=` directly to make_predictions
#   use_compound_names -- whether the input carries an extra name column
#   preds_path         -- where predictions would be written (not used in smiles mode)
#   checkpoint_dir     -- folder with the trained IR model(s); this is the model
#                         that actually produces the spectra
#   spectra_type       -- "experimental" vs "simulated": selects the input parser
#   spectra_type_nr    -- numeric id of that parser (computed by modify_predict_args)
#   checkpoint_path    -- single-file override; None = use every file in checkpoint_dir
#   batch_size         -- molecules per forward pass
#   no_cuda            -- nested list: one flag per ensemble member
#   features_generator -- extra descriptors (None = spectra only)
#   features_path      -- pre-computed descriptors (None = none)
#   max_data_size      -- truncate the prediction set (kept at 100 here, but the
#                         batching loop in cell 8 ignores it and feeds all molecules)
#   ensemble_variance  -- also emit predictive variance (needs >1 checkpoint)
#   ensemble_variance_conv -- variance threshold for that feature (0.0 = unused)
#
# NOTE (known issue, left unchanged): the relative paths are resolved against
# os.getcwd() inside chemprop, so the notebook must be started from the repo root.
# =============================================================================

config_dict = {
    "gpu": list(range(torch.cuda.device_count())),
    "test_path": ["./models/chemprop-ir/ir_models_data/solvation_example/solvation_spectra.csv"],
    "use_compound_names": [False],
    "preds_path": ["./models/chemprop-ir/ir_models_data/ir_preds_test_2.csv"],
    "checkpoint_dir": ["./models/chemprop-ir/ir_models_data/experiment_model/model_files"],
    "spectra_type": ["experimental"],
    "spectra_type_nr": [0],
    "checkpoint_path": [None],
    "batch_size": [50],
    "no_cuda": [[False]],
    "features_generator": [None],
    "features_path": [None],
    "max_data_size": [100],
    "ensemble_variance": [False],
    "ensemble_variance_conv": [0.0],
}

from argparse import Namespace

def parse_arguments(hyperparameters: dict) -> Namespace:
    """Unwrap a dict of one-element lists into an attribute-style Namespace.

    Args:
        hyperparameters: mapping key -> [value]; one value per key.

    Returns:
        Namespace(**{key: value}), i.e. `args.batch_size` instead of
        `config_dict["batch_size"][0]`.
    """
    parsed_args = {key: val[0] for key, val in hyperparameters.items()}
    return Namespace(**parsed_args)

# `args` is the object handed to chemprop; cell 4 mutates it in place
args = parse_arguments(config_dict)

In [ ]:
# =============================================================================
# CELL 4 -- Fill the derived chemprop arguments
#
# `modify_predict_args(args)` is chemprop's normalisation step and mutates the
# Namespace **in place** (it returns None). What it adds that later code relies on:
#   * spectra_type_nr  -- numeric id of the spectra parser chosen by spectra_type
#   * cuda / device    -- resolved from `no_cuda` and `gpu`
#   * feature sizes    -- input dimensionality implied by features_generator/path
#   * sets of valid feature generators / activation functions
#
# Because it is in-place, `make_predictions(args, smiles=...)` in cell 7 can be
# called with the same object afterwards; do NOT rebuild `args` from
# `config_dict` after this call without calling modify_predict_args again.
# =============================================================================

from chemprop.train import make_predictions
from chemprop.parsing import modify_predict_args

modify_predict_args(args)   # returns None; all changes land in `args`

### Simulate IR for big ZINC dataset

> **NOTE (stale heading, left unchanged):** this section no longer targets ZINC.
> Cell 8 reads `data/IBM_dataset/IBM_SMI_data.csv` and writes to
> `data/IBM_dataset/IBM_ir_sim`, because that folder is what
> `1.0_Experiment_Notebook___.ipynb` consumes for the IBM benchmark. The ZINC
> variant would only differ in the input CSV (the large ZINC SMILES list) and the
> output folder.
>
> The cell that follows (6) repeats the imports of cells 1-2 and cell 7 repeats
> the config of cell 3, so that this section can be run on a fresh kernel without
> scrolling up. Re-running them is harmless (imports are cached, the dict is rebuilt).

In [ ]:
# =============================================================================
# CELL 6 -- Standalone bootstrap for this section (duplicate of cells 1-2)
#
# Kept so that a reader can jump straight to "Simulate IR" on a fresh kernel.
# Everything here is already imported above; re-running only appends the
# chemprop-IR directory to sys.path a second time (harmless, see cell 1).
#
# Additional imports compared with cell 2:
#   pandas         -- reading the SMILES CSV and writing the per-molecule IR CSVs
#   rdkit.Chem     -- needed by calculate_molecular_formulas() in cell 7
# =============================================================================

import os
import sys
from pathlib import Path

# Get the current working directory (where the notebook is running)
notebook_dir = Path.cwd()

# Add the chemprop-IR directory to the path
sys.path.append(str(notebook_dir / "chemprop-IR"))

from argparse import Namespace
import csv
from typing import List, Optional

import numpy as np
import torch
from tqdm import tqdm

# Import chemprop modules
from chemprop.train.predict import predict
from chemprop.data import MoleculeDataset
from chemprop.data.utils import get_data, get_data_from_smiles
from chemprop.utils import load_args, load_checkpoint, load_scalers
from chemprop.train.spectral_loss import roundrobin_sid
from chemprop.features import get_available_features_generators
get_available_features_generators()   # display-only

from chemprop.train import make_predictions
from chemprop.parsing import modify_predict_args
import pandas as pd
from rdkit import Chem

In [ ]:
# =============================================================================
# CELL 7 -- Config (duplicate of cell 3) + the two helpers used by the loop
#
# The helpers:
#   generate_spectral_data_batch(args, smiles)
#       Wrapper around chemprop's make_predictions. It returns `avg_preds`, the
#       ensemble-averaged descriptor vector(s), and the per-model DataFrame.
#       Despite the name it is NOT a "batch generator" -- it is the prediction.
#
#   calculate_molecular_formulas(smiles_list)
#       RDKit formula string (e.g. "C6H6") for each SMILES; used to build the
#       MF column of the summary table in cell 9.
#
# NOTE (known issue, left unchanged): generate_spectral_data_batch ignores its
#   `smiles` argument and instead reads the module-global `smiles_list`. That
#   global is only created inside the loop of cell 8, so the function works
#   *by accident* during that loop and raises NameError anywhere else (e.g.
#   calling it interactively before cell 8). The `smiles=smiles_list` argument
#   should be `smiles=smiles`.
#
# NOTE (known issue, left unchanged): `predictions_df` is assigned but never
#   used, and the TODO about "neural network-based spectral data generation"
#   is stale -- make_predictions already runs the trained chemprop network.
#
# NOTE (robustness, left unchanged): calculate_molecular_formulas does not guard
#   against `Chem.MolFromSmiles` returning None (invalid SMILES), which would make
#   CalcMolFormula raise. One bad SMILES therefore aborts the whole batch in cell 8.
# =============================================================================

config_dict = {
    "gpu": list(range(torch.cuda.device_count())),
    "test_path": ["./models/chemprop-ir/ir_models_data/solvation_example/solvation_spectra.csv"],
    "use_compound_names": [False],
    "preds_path": ["./models/chemprop-ir/ir_models_data/ir_preds_test_2.csv"],
    "checkpoint_dir": ["./models/chemprop-ir/ir_models_data/experiment_model/model_files"],
    "spectra_type": ["experimental"],
    "spectra_type_nr": [0],
    "checkpoint_path": [None],
    "batch_size": [50],
    "no_cuda": [[False]],
    "features_generator": [None],
    "features_path": [None],
    "max_data_size": [100],
    "ensemble_variance": [False],
    "ensemble_variance_conv": [0.0],
}



def parse_arguments(hyperparameters: dict) -> Namespace:
    """Unwrap a dict of one-element lists into a Namespace (same as cells 3/4).

    Args:
        hyperparameters: mapping key -> [value].

    Returns:
        Namespace with attributes set to the unwrapped values.
    """
    parsed_args = {key: val[0] for key, val in hyperparameters.items()}
    return Namespace(**parsed_args)


def generate_spectral_data_batch(args: Namespace, smiles: list) -> np.ndarray:
    """Run the chemprop-IR ensemble on a batch of SMILES.

    Args:
        args: chemprop Namespace *after* modify_predict_args.
        smiles: list of SMILES strings (NOTE: currently unused -- see header).

    Returns:
        avg_preds: ensemble-averaged prediction array, shape (n_molecules, n_outputs),
        where n_outputs is the predicted IR spectrum length (1000 points).
    """
    avg_preds, predictions_df = make_predictions(args, smiles=smiles_list)
    # TODO: Implement the neural network-based spectral data generation
    return avg_preds

def calculate_molecular_formulas(smiles_list: list) -> list:
    """Return the RDKit molecular formula for every SMILES in the list.

    Args:
        smiles_list: iterable of SMILES strings.

    Returns:
        List of formula strings such as "C6H6" / "C12H22O11", same order as input.
    """
    formulas = []
    for smiles in smiles_list:
        # Parse the SMILES into an RDKit molecule (None if the string is invalid)
        molecule = Chem.MolFromSmiles(smiles)
        # rdMolDescriptors is reachable through Chem without an explicit import
        formula = Chem.rdMolDescriptors.CalcMolFormula(molecule)
        formulas.append(formula)
    return formulas

args = parse_arguments(config_dict)
modify_predict_args(args)   # in-place; fills spectra_type_nr / device flags

In [ ]:
# =============================================================================
# CELL 8 -- Main loop: SMILES -> 1000-point IR spectrum -> one CSV per molecule
#
# Flow per iteration (batch of 64 molecules):
#   1. slice the input DataFrame
#   2. pull the SMILES and the sample-ids
#   3. make_predictions -> (n_batch, 1000) matrix of ensemble-averaged spectra
#   4. write data/IBM_dataset/IBM_ir_sim/<sample-id>.csv, one column "spectra"
#   5. compute the molecular formula of each SMILES and accumulate the three
#      summary lists that cell 9 turns into a DataFrame
#
# Why one CSV per molecule:
#   config.IR_data_folder is read by the MMST dataloader per sample, so the IR
#   modality has to be addressable per molecule id rather than stored in a single
#   wide table.
#
# NOTE (known issue, left unchanged): `smiles_list.extend(smiles_list)` extends
#   the list with ITSELF, so every batch is duplicated (64 -> 128 entries) while
#   `sample_id_list` and `molecular_formulas` grow by only 64. After the first
#   batch the three columns of cell 9's DataFrame therefore have different
#   lengths and pandas raises "ValueError: arrays must all be same length".
#   The intended line is `smiles_all.extend(smiles_list)`. The per-molecule CSV
#   writing (the part this notebook needs for §1.0) is unaffected -- it happens
#   earlier in the loop.
#
# NOTE (observation, left unchanged): the loop variable `smiles_list` shadows the
#   name used by generate_spectral_data_batch() in cell 7. That shadowing is the
#   only reason the helper works inside this loop.
#
# NOTE (step numbering, left unchanged): the comments mention "Step 2 and 3" and
#   "Step 5" although there is no Step 1 or 4 -- leftovers of a longer draft.
# =============================================================================

base_dir = os.getcwd()
csv_file = os.path.join(base_dir,"data/IBM_dataset/IBM_SMI_data.csv")
df = pd.read_csv(csv_file)

# Create a directory to save the spectral data files
output_directory = os.path.join(base_dir,"data/IBM_dataset/IBM_ir_sim")
if not os.path.exists(output_directory):
    os.makedirs(output_directory)

molecular_formulas = []   # formula strings, one per input molecule (summary table, cell 9)
smiles_list = []          # NOTE: immediately replaced by the loop variable below
sample_id_list = []       # ids, one per input molecule (summary table, cell 9)

batch_size = 64           # molecules per chemprop forward pass (NOT args.batch_size=50)

# Step 2 and 3: Process in batches of 64
for i in tqdm(range(0, len(df), batch_size)):
    batch = df.iloc[i:i+batch_size]              # the i-th slice, <= 64 rows
    smiles_list = batch['SMILES'].tolist()       # local per-batch SMILES
    sample_ids = batch['sample-id'].tolist()     # matching ids, same order

    # Generate spectral data for the batch
    # -> shape (len(batch), 1000); one row per molecule
    spectral_data_batch = generate_spectral_data_batch(args, smiles_list)


    # Save spectral data for each sample in the batch
    for j, spectral_data in enumerate(spectral_data_batch):
        # spectral_data is the 1000-point vector; DataFrame({'spectra': ...}) gives
        # a single-column, 1000-row CSV
        output_file_path = os.path.join(output_directory, f"{sample_ids[j]}.csv")
        pd.DataFrame({'spectra': spectral_data}).to_csv(output_file_path, index=False)


    # Step 5: Calculate molecular formula and append to list
    formula = calculate_molecular_formulas(smiles_list)
    molecular_formulas.extend(formula)
    # NOTE (bug, see header): self-extension doubles the batch in `smiles_list`
    smiles_list.extend(smiles_list)
    sample_id_list.extend(sample_ids)



In [ ]:
# =============================================================================
# CELL 9 -- Write the summary table chemprop-ir-sim-IBM.csv
#
# Columns:
#   SMILES     -- the input SMILES
#   sample-id  -- molecule id, matching the file name in IBM_ir_sim/
#   formula    -- RDKit molecular formula (the "MF" modality)
#
# This CSV is the *index* of the per-molecule IR files: downstream code uses the
# id to locate `<id>.csv` inside config.IR_data_folder.
#
# NOTE (known issue, left unchanged): because of the self-extending list in cell 8,
#   `smiles_list` can be roughly twice as long as `sample_id_list` /
#   `molecular_formulas`, in which case this DataFrame construction raises
#   ValueError. If you need the summary table, fix cell 8 first (use a separate
#   accumulator list instead of `smiles_list.extend(smiles_list)`).
# =============================================================================

# Save the accumulated data into a DataFrame
results_df = pd.DataFrame({
    'SMILES': smiles_list,
    'sample-id': sample_id_list,
    'formula': molecular_formulas
})

# Step 6: Save the updated DataFrame to a new CSV file
new_csv_path = os.path.join(base_dir,"data/IBM_dataset/chemprop-ir-sim-IBM.csv")

results_df.to_csv(new_csv_path, index=False)

In [ ]:
# =============================================================================
# CELL 10 -- Sanity check: how many IR files were produced?
#
# Should equal the number of molecules in IBM_SMI_data.csv. A smaller number means
# molecules were skipped (invalid SMILES / failed prediction), and those samples
# will have an empty IR channel in §1.0 (config.IR_data_folder).
# =============================================================================

import os
output_directory = os.path.join(base_dir,"data/IBM_dataset/IBM_ir_sim")
len(os.listdir(output_directory))   # cell output: number of per-molecule CSVs